# 07 — Memory audit of the mock HybridSN workflow

Where does memory rise between opening the mock cube and taking the first training step? Run all cells in a fresh CPU kernel after notebooks 03 and 04. This profiles one pass through the training side of notebook 06, using its model, ten whitened PCA components, 9 × 9 patches and batch size of 64.

In [1]:
from pathlib import Path

import h5py
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

root = Path(".") if Path("notebooks").is_dir() else Path("..")
cube_file = root / "data/mock/mock_cube.h5"
pca_file = root / "data/processed/mock_pca.npz"
split_row = 38  # First validation row in notebook 06

import platform
import threading
from contextlib import contextmanager

import psutil
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

torch.manual_seed(0)
MiB = 2 ** 20
process = psutil.Process()
measurements = []
print(platform.system(), platform.machine(), "| torch", torch.__version__, "| CPU threads", torch.get_num_threads())

Windows AMD64 | torch 2.15.0+cpu | CPU threads 4


### Measure process memory

For each stage, sample the kernel's resident set size (RSS) at entry, exit and approximately every 5 ms. RSS includes the interpreter, libraries, native buffers and any memory retained from earlier stages. The observed peak is a lower bound because short allocations can occur between samples. Run the stages in order.

In [2]:
@contextmanager
def measure(stage):
    start = process.memory_info().rss
    samples = [start]
    stop = threading.Event()

    def sample():
        while not stop.wait(0.005):
            samples.append(process.memory_info().rss)

    monitor = threading.Thread(target=sample, daemon=True)
    monitor.start()
    try:
        yield
    finally:
        end = process.memory_info().rss
        stop.set()
        monitor.join()
        measurements.append((stage, start, max(max(samples), end), end))

### Load reflectance and fit the training transform

Read the full HDF5 cube as notebook 06 does, apply the wavelength filter from 04, and fit scaling and PCA on valid training rows. The float64 training arrays and the full SVD workspace contribute to process memory. Keep the validation region out of the fit.

In [3]:
with measure("Read cube and select bands"):
    with np.load(pca_file) as archive:
        valid = archive["valid"]
        kept = archive["wavelengths_kept"]
    with h5py.File(cube_file) as source:
        cube = source["cube"][...]
        labels = source["labels"][...]
        bands = np.isin(source["wavelengths_nm"][...], kept)
    spectra = cube[..., bands]
    H, W, B = cube.shape
    cube_bytes = cube.nbytes
    del cube

K, patch, batch_size = 10, 9, 64
with measure("Fit PCA and transform training rows"):
    region_valid = valid[:split_row]
    train_spectra = spectra[:split_row][region_valid].astype(np.float64)
    scaler = StandardScaler().fit(train_spectra)
    scaled = scaler.transform(train_spectra)
    pca = PCA(n_components=K, whiten=True, svd_solver="full").fit(scaled)
    scores = np.zeros((*region_valid.shape, K), dtype=np.float32)
    scores[region_valid] = pca.transform(scaled)

print(f"Cube {H} x {W} x {B} | {bands.sum()} retained bands | {len(train_spectra)} training pixels")

Cube 67 x 67 x 200 | 166 retained bands | 2546 training pixels


### Build patch views and take one CPU training step

The dataset and network definitions below match notebook 06. Reflection at the training boundary keeps validation pixels outside every training patch. The default single-process loader copies requested patches into one contiguous batch.

In [4]:
class PatchDataset(Dataset):
    def __init__(self, windows, centres, labels):
        self.windows = windows            # (H, W, K, P, P) view, no copy
        self.centres = centres            # (n, 2) array of (row, col)
        self.labels = labels              # (n,) 0 = healthy, 1 = stressed

    def __len__(self):
        return len(self.centres)

    def __getitem__(self, i):
        r, c = self.centres[i]
        patch = torch.from_numpy(np.ascontiguousarray(self.windows[r, c]))     # (K, P, P)
        return patch.unsqueeze(0), torch.tensor(float(self.labels[i]))


class HybridSNMock(nn.Module):
    def __init__(self, n_bands=10, patch=9, dropout=0.4):
        super().__init__()
        self.spectral_spatial = nn.Sequential(                 # 3D part
            nn.Conv3d(1, 8, kernel_size=3), nn.ReLU(),
            nn.Conv3d(8, 16, kernel_size=3), nn.ReLU(),
            nn.Conv3d(16, 32, kernel_size=3), nn.ReLU(),
        )
        with torch.no_grad():                                  # measure the shape instead of computing it by hand
            d, h, w = self.spectral_spatial(torch.zeros(1, 1, n_bands, patch, patch)).shape[2:]

        self.spatial = nn.Sequential(                          # 2D part
            nn.Conv2d(32 * d, 64, kernel_size=3), nn.ReLU(),
        )
        with torch.no_grad():
            flat = self.spatial(torch.zeros(1, 32 * d, h, w)).numel()

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1),                                  # ONE logit: "stressed"
        )

    def forward(self, x, verbose=False):
        show = (lambda name, t: print(f"{name:<28}{tuple(t.shape)}")) if verbose else (lambda name, t: None)
        show("input", x)
        x = self.spectral_spatial(x)                           # (N, 32, D', H', W')
        show("after 3 x Conv3d", x)
        n, c, d, h, w = x.shape
        x = x.reshape(n, c * d, h, w)                          # merge channels and bands
        show("reshaped for 2D", x)
        x = self.spatial(x)
        show("after Conv2d", x)
        x = self.classifier(x).squeeze(1)
        show("output (one logit each)", x)
        return x

In [5]:
with measure("Pad training region and form view"):
    radius = patch // 2
    padded = np.pad(scores, ((radius, radius), (radius, radius), (0, 0)), mode="reflect")
    windows = np.lib.stride_tricks.sliding_window_view(padded, (patch, patch), axis=(0, 1))
    assert np.shares_memory(windows, padded)
    dataset = PatchDataset(windows, np.argwhere(region_valid), labels[:split_row][region_valid])

with measure("Collate 64 patches"):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    xb, yb = next(iter(loader))
    assert tuple(xb.shape) == (batch_size, 1, K, patch, patch)

with measure("Create network and Adam"):
    model = HybridSNMock(n_bands=K, patch=patch)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    healthy, stressed = np.bincount(dataset.labels)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(healthy / stressed, dtype=torch.float32))

with measure("First forward, backward and update"):
    model.train()
    optimizer.zero_grad()
    loss = criterion(model(xb), yb)
    loss.backward()
    optimizer.step()
assert torch.isfinite(loss) and len(optimizer.state) > 0

### Observed RSS and array storage

Read the peak column for the largest working set. The increase above stage entry shows how much additional RSS was observed during that stage. These increases cannot be added to obtain a total.

In [6]:
print(f"{'Stage':38} {'Entry':>9} {'Peak':>9} {'Exit':>9} {'Peak-entry':>11}  MiB")
for stage, start, peak, end in measurements:
    print(f"{stage:38} {start/MiB:9.1f} {peak/MiB:9.1f} {end/MiB:9.1f} {(peak-start)/MiB:11.1f}")
print()
print(f"Full reflectance cube: {cube_bytes/MiB:.3f} MiB")
print(f"Padded training scores backing the view: {padded.nbytes/MiB:.3f} MiB")
print(f"All training windows if copied: {windows.nbytes/MiB:.3f} MiB")
print(f"One input batch: {xb.numel()*xb.element_size()/MiB:.3f} MiB")

Stage                                      Entry      Peak      Exit  Peak-entry  MiB
Read cube and select bands                 302.1     314.1     313.8        12.0
Fit PCA and transform training rows        313.9     327.2     323.3        13.4
Pad training region and form view          323.4     323.5     323.5         0.1
Collate 64 patches                         323.5     324.9     324.9         1.4
Create network and Adam                    322.3     377.9     377.9        55.7
First forward, backward and update         377.9     390.2     390.2        12.3

Full reflectance cube: 3.425 MiB
Padded training scores backing the view: 0.132 MiB
All training windows if copied: 7.867 MiB
One input batch: 0.198 MiB


### Estimate a larger scene

Use a hypothetical 1,000 × 1,000 scene at the same spectral and spatial resolution in array terms. Each estimate is a single float32 array payload. Process overhead, masks, PCA workspaces, gradients, activations and optimizer state add memory. The fixed-size batch remains the same size as the scene grows.

In [7]:
side = 1000
itemsize = scores.itemsize
for name, elements in [
    ("Raw reflectance cube", side**2 * B),
    ("Ten-component scores", side**2 * K),
    ("All patches copied", side**2 * K * patch**2),
    ("One 64-patch batch", batch_size * K * patch**2),
]:
    print(f"{name:24} {elements * itemsize / MiB:10.3f} MiB")

Raw reflectance cube        762.939 MiB
Ten-component scores         38.147 MiB
All patches copied         3089.905 MiB
One 64-patch batch            0.198 MiB


For larger scenes, read spatial windows from HDF5, transform blocks with the fitted training preprocessing, and keep patch extraction on demand. A four-pixel halo provides the neighbours for centres at an interior tile edge. Measure a representative tile and batch before choosing an operating memory budget.